# Práctico de DeepFM

# 1. Introducción

En este práctico implementaremos el modelo **DeepFM**, propuesto en el paper [DeepFM: A Factorization-Machine based Neural Network for CTR Prediction](https://arxiv.org/abs/1703.04247).   Para ello utilizaremos la librería **DeepCTR-torch**, que incluye distintos modelos de recomendación basados en redes neuronales.  

El dataset a utilizar será **MovieLens-100k**.  

En cuanto a la evaluación, algunas métricas (como *Precision@K* y *Recall@K*) serán implementadas manualmente, mientras que otras métricas se calcularán con **DeepCTR-torch**.

# 2. Instalación de librerías

In [37]:
!pip install deepctr-torch torch pandas numpy scikit-learn recommenders

# 3. Recomendación de películas sin el género

Importar librerías

In [38]:
from recommenders.datasets import movielens
from deepctr_torch.inputs import SparseFeat, VarLenSparseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd

Descargar dataset

In [3]:
df = movielens.load_pandas_df(
    size="100k",
    genres_col="genre",
    header=["userID", "itemID", "rating"]
)

print(df.shape)
df.sample(5, random_state=42)

100%|██████████| 4.81k/4.81k [00:00<00:00, 12.7kKB/s]


(100000, 4)


,userID,itemID,rating,genre
75721,877,381,4.0,Comedy|Romance
80184,815,602,3.0,Musical|Romance
19864,94,431,4.0,Action|Adventure
76699,416,875,2.0,Drama|Romance
92991,500,182,2.0,Crime|Drama


En **DeepCTR-torch**, el modelo DeepFM está diseñado originalmente para predecir *Click-Through Rate (CTR)*, por lo que la variable objetivo debe ser **binaria**:  
- `0` si el usuario no realizó la acción (no clickeó),  
- `1` si el usuario sí la realizó (clickeó).  

En el caso del dataset de MovieLens, utilizamos las calificaciones como aproximación al interés del usuario. Como el promedio de ratings es cercano a **3.5**, definimos el umbral igual a **4**:  
- ratings `≥ 4` se consideran interacciones positivas (`1`),  
- ratings `< 4` se consideran negativas (`0`).


In [4]:
df["rating"].mean()

np.float64(3.52986)

In [5]:
df = df.copy()
df["label"] = (df["rating"] >= 4).astype(int)

Seleccionamos el primer género como el género principal de la película y el que nos ayudará a la recomendación

In [6]:
def primary_genre(g):
    if isinstance(g, str):
        return g.split("|")[0] if "|" in g else g
    return "Unknown"
df["primary_genre"] = df["genre"].apply(primary_genre).fillna("Unknown")

Se crean diccionarios que convierten cada identificador de usuario, película y género a un índice entero.

Esto es necesario porque los modelos como DeepFM trabajan con índices enteros que luego son pasados a embeddings.

In [7]:
uid2idx = {u:i for i, u in enumerate(sorted(df["userID"].unique()))}
iid2idx = {m:i for i, m in enumerate(sorted(df["itemID"].unique()))}
gid2idx = {g:i for i, g in enumerate(sorted(df["primary_genre"].unique()))}

Se reemplazan los IDs originales por sus índices correspondientes.

In [8]:
df["user_id"] = df["userID"].map(uid2idx).astype(int)
df["item_id"] = df["itemID"].map(iid2idx).astype(int)
df["genre_id"] = df["primary_genre"].map(gid2idx).astype(int)

Printeamos el número total de usuarios, películas y géneros únicos.

In [9]:
n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()
n_genres = df["genre_id"].nunique()
print(n_users, n_items, n_genres)

943 1682 19


Dividimos el dataset en tres subconjuntos:

- **Entrenamiento (80%)**: se utiliza para entrenar el modelo
- **Validación (10%)**: permite evaluar el desempeño durante el entrenamiento y ajustar hiperparámetros.  
- **Test (10%)**: se reserva para la evaluación final de métricas

In [10]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df  = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df["label"])

En esta celda definimos las **features categóricas** que usará el modelo:

- **`user_id`**: representa a cada usuario del dataset.  
- **`item_id`**: representa a cada ítem o película.  
- **`genre_id`**: representa el género principal de cada película.  

Cada una se declara como un `SparseFeat`, lo que significa que serán tratadas como **variables categóricas** y proyectadas a un **embedding de x dimensiones**.  
De esta manera, el modelo aprende representaciones densas (vectores) tanto para usuarios como para ítems y géneros, capturando relaciones latentes entre ellos.


In [11]:
user_feat = SparseFeat("user_id",  vocabulary_size=n_users,  embedding_dim=16)
item_feat = SparseFeat("item_id",  vocabulary_size=n_items,  embedding_dim=16)

En esta celda se definen las columnas de features para cada parte del modelo:

- **`linear_feature_columns`**: features que alimentan la parte **lineal** (Factorization Machine).  
- **`dnn_feature_columns`**: features que alimentan la parte **no lineal** (red neuronal profunda, DNN).  

En este caso, ambas listas contienen las **mismas variables** (`user_id`, `item_id`, `genre_id`).  
Esto significa que tanto la parte lineal como la no lineal del modelo trabajarán sobre la **misma información de entrada**, pero cada componente la procesará de manera distinta.  

Finalmente, con **`get_feature_names`** se obtiene la lista consolidada de todas las features, necesaria para preparar los datos de entrada al modelo.

In [12]:
linear_feature_columns = [user_feat, item_feat]
dnn_feature_columns = [user_feat, item_feat]
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

Se define la función **`build_X`**, que recibe un DataFrame y devuelve un diccionario con las columnas especificadas en `feature_names`.  
  - Cada clave del diccionario corresponde al nombre de una feature (`user_id`, `item_id`, `genre_id`).  
  - Cada valor es un array con los índices de esa feature.  
  - Este formato es el requerido por **DeepCTR-torch** para entrenar el modelo.


In [13]:
def build_X(df_):
    return {name: df_[name].values for name in feature_names if name in df_.columns}

Luego se construyen los conjuntos de datos:  
- **`X_train`, `y_train`**: features y etiquetas del conjunto de entrenamiento.  
- **`X_val`, `y_val`**: features y etiquetas del conjunto de validación.  
- **`X_test`, `y_test`**: features y etiquetas del conjunto de testeo.

De esta forma, los datos quedan listos para ser entregados al modelo DeepFM.

In [14]:
X_train, y_train = build_X(train_df), train_df["label"].values
X_val, y_val = build_X(val_df), val_df["label"].values
X_test, y_test = build_X(test_df), test_df["label"].values

En esta celda se instancia el modelo **DeepFM** usando la implementación de *DeepCTR-torch*.  
Los parámetros principales son:

- **`linear_feature_columns` y `dnn_feature_columns`**: definen las features de entrada para la parte lineal (FM) y para la red neuronal (DNN).  
- **`task='binary'`**: indica que el modelo resolverá un problema de clasificación binaria (en este caso, interacción relevante o no relevante).  
- **`dnn_hidden_units=(128, 64)`**: la parte de red neuronal profunda tendrá dos capas ocultas, la primera con 128 neuronas y la segunda con 64.  
- **`dnn_dropout=0.2`**: aplica *dropout* del 20% en las capas de la DNN para reducir sobreajuste.  
- **`device="cpu"`**: especifica que el entrenamiento se realizará en CPU (podría configurarse `"cuda"` para usar GPU).

De esta forma, el modelo DeepFM combina la parte **lineal (Factorization Machines)** y la parte **no lineal (red neuronal profunda)** en un solo modelo de recomendación.


In [15]:
model = DeepFM(
    linear_feature_columns, dnn_feature_columns,
    task='binary', dnn_hidden_units=(128,64), dnn_dropout=0.2, device="cpu"
)

En esta celda se compila el modelo DeepFM, definiendo cómo será entrenado:

- **`optimizer="adam"`**: se utiliza el [optimizador Adam](https://arxiv.org/abs/1412.6980).
- **`loss="binary_crossentropy"`**: la función de pérdida adecuada para problemas de clasificación binaria.
- **`metrics=["auc"]`**: además de la pérdida, se calculará el **AUC**. Esta métrica es común en tareas de predicción de CTR, ya que mide la capacidad del modelo para rankear correctamente ejemplos positivos frente a negativos.

En **DeepCTR-torch**, las métricas que pueden configurarse en `metrics=[...]` son: "auc", "mse" y ,"accuracy". Más información en el método _get_metrics de la [documentación](https://deepctr-torch.readthedocs.io/en/latest/_modules/deepctr_torch/models/basemodel.html#BaseModel.compile).

De esta forma, se establecen tanto la función de optimización como las métricas de evaluación que se usarán durante el entrenamiento del modelo.


In [16]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

En esta celda se entrena el modelo DeepFM con el método **`fit`**, indicando los parámetros de entrenamiento:

- **`X_train, y_train`**: datos de entrada y etiquetas del conjunto de entrenamiento.  
- **`batch_size=2048`**: cantidad de muestras que se procesan en cada iteración antes de actualizar los parámetros.  
- **`epochs=10`**: número de veces que el modelo recorre por completo el conjunto de entrenamiento.  
- **`verbose=2`**: controla el nivel de detalle mostrado durante el entrenamiento (2 muestra una línea por época).  
- **`validation_data=(X_val, y_val)`**: conjunto de validación que se evalúa al final de cada época para monitorear el desempeño del modelo y detectar sobreajuste.

In [17]:
model.fit(X_train, y_train, batch_size=2048, epochs=10, verbose=2, validation_data=(X_val, y_val))

cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/10
6s - loss:  0.6838 - auc:  0.6202 - val_auc:  0.7557
Epoch 2/10
2s - loss:  0.6449 - auc:  0.7698 - val_auc:  0.7598
Epoch 3/10
2s - loss:  0.5861 - auc:  0.7818 - val_auc:  0.7665
Epoch 4/10
3s - loss:  0.5634 - auc:  0.7921 - val_auc:  0.7695
Epoch 5/10
1s - loss:  0.5537 - auc:  0.7949 - val_auc:  0.7707
Epoch 6/10
2s - loss:  0.5486 - auc:  0.7957 - val_auc:  0.7712
Epoch 7/10
3s - loss:  0.5456 - auc:  0.7975 - val_auc:  0.7723
Epoch 8/10
1s - loss:  0.5439 - auc:  0.7986 - val_auc:  0.7724
Epoch 9/10
1s - loss:  0.5425 - auc:  0.7995 - val_auc:  0.7730
Epoch 10/10
1s - loss:  0.5412 - auc:  0.7986 - val_auc:  0.7733


Definimos las funciones para evaluación Precision@K y Recall@K

In [18]:
def precision_recall_at_k(df_user, k):
    df_user = df_user.sort_values('score', ascending=False)
    topk = df_user.head(k)
    hits_k = int(topk['label'].sum())
    total_rel = int(df_user['label'].sum())

    prec = hits_k / k
    rec = hits_k / total_rel if total_rel > 0 else np.nan

    return prec, rec, total_rel

def macro_precision_recall_at_k(df, k):
    precs, recs = [], []
    for uid, df_u in df.groupby('user_id'):
        p, r, tot_rel = precision_recall_at_k(df_u, k)
        precs.append(p)
        if not np.isnan(r):
            recs.append(r)

    return float(np.mean(precs)), (float(np.mean(recs)) if len(recs) > 0 else np.nan)

Se generan las predicciones del modelo sobre el conjunto de **test**:

- **`model.predict(X_test, batch_size=4096)`**: obtiene los puntajes de predicción para cada par usuario–ítem del conjunto de test.  
- **`.reshape(-1)`**: ajusta el formato de salida a un vector unidimensional.  
- **`test_eval = test_df[['user_id','item_id','label']].copy()`**: se crea un DataFrame de evaluación que contiene los identificadores de usuario, ítem y la etiqueta real (`label`).  
- **`test_eval['score'] = y_scores`**: se agrega una nueva columna con los puntajes predichos por el modelo.

In [19]:
y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
test_eval = test_df[['user_id','item_id','label']].copy()
test_eval['score'] = y_scores

Calculamos las métricas para K = 5, 10 y 20. Guardamos sus resultados en un dataframe para mostrarlo más fácilmente

In [20]:
K_list = [5, 10, 20]
rows = []
for K in K_list:
    p, r = macro_precision_recall_at_k(test_eval, K)
    rows.append({
        'K': K,
        'Precision@K (macro)': p,
        'Recall@K (macro)': r,
    })

metrics_df = pd.DataFrame(rows)
metrics_df

,K,Precision@K (macro),Recall@K (macro)
0,5,0.558887,0.702980
1,10,0.418522,0.871136
2,20,0.270610,0.971696


# 4. Recomendación de películas agregando el género

Se define la función **`split_genres`**, que procesa la columna de géneros de cada película:

- Si el valor es un string válido (`"Comedy|Romance|Drama"`), lo divide en una **lista de géneros individuales**, separando por el carácter `"|"`.  
- Si el valor es nulo o no es un string, asigna el género `"Unknown"`.  

De esta forma, cada película queda representada no por un único género principal, sino por una **lista de todos sus géneros asociados**.

In [21]:
def split_genres(g):
    if isinstance(g, str) and g.strip():
        return [x.strip() for x in g.split("|")]
    return ["Unknown"]

In [22]:
df["genres_list"] = df["genre"].apply(split_genres)

Se construye el **vocabulario de géneros** y se define el esquema de codificación numérica:

- **`all_genres`**: contiene la lista ordenada de todos los géneros únicos presentes en el dataset.  
- **`genre2id`**: asigna a cada género un identificador numérico entero, comenzando en `1`.  
  - El `+1` se utiliza porque el índice `0` se reserva como **padding**.  
- **`PAD_ID = 0`**: valor especial que representa el padding y se usa para completar secuencias de géneros a una longitud fija.  
- **`VOCAB_SIZE_GENRE = len(all_genres) + 1`**: tamaño total del vocabulario, incluyendo la posición adicional para el padding.

De esta forma, cada género puede mapearse a un número entero, y las secuencias de géneros pueden representarse con longitudes uniformes.

In [23]:
all_genres = sorted({g for lst in df["genres_list"] for g in lst})
genre2id = {g: i+1 for i, g in enumerate(all_genres)}
PAD_ID = 0
VOCAB_SIZE_GENRE = len(all_genres) + 1

In [24]:
df["genres_ids"] = df["genres_list"].apply(lambda lst: [genre2id[g] for g in lst])

Se define la función **`pad_seq`**, utilizada para uniformar la longitud de las secuencias de géneros:

- **`MAXLEN = 5`**: se fija la longitud máxima de las secuencias.  
- La función **`pad_seq(ids, maxlen, pad_value)`** recibe una lista de identificadores de géneros:  
  - Si la lista es más larga que `MAXLEN`, se **trunca** a la longitud máxima.  
  - Si es más corta, se **rellena con el valor de padding (`PAD_ID = 0`)** hasta alcanzar `MAXLEN`.  

De esta forma, todas las películas quedan representadas con una secuencia de géneros de tamaño fijo (`MAXLEN`), lo que permite procesarlas en `deepctr-torch`.

In [25]:
MAXLEN = 3

In [26]:
def pad_seq(ids, maxlen=MAXLEN, pad_value=PAD_ID):
    ids = ids[:maxlen]
    return ids + [pad_value] * (maxlen - len(ids))

Se crean las columnas con el padding, "genres_seq_len" almacena la longitud real de cada secuencia

In [27]:
df["genres_seq"] = df["genres_ids"].apply(lambda ids: pad_seq(ids, MAXLEN, PAD_ID))
df["genres_seq_len"] = df["genres_ids"].apply(lambda ids: min(len(ids), MAXLEN))

En esta celda se define el feature secuencial de géneros y se incorporan todas las features al modelo:

- Se utiliza `VarLenSparseFeat` para representar la lista de géneros de cada película como una secuencia.  
- **`SparseFeat("genres_seq", vocabulary_size=VOCAB_SIZE_GENRE, embedding_dim=16)`**: cada género se convierte en un embedding de 16 dimensiones.  
- **`maxlen=MAXLEN`**: fija la longitud máxima de la secuencia (con padding si es necesario).  
- **`combiner="mean"`**: combina los embeddings de todos los géneros de una película calculando su promedio.  
- **`length_name="genres_seq_len"`**: indica la longitud real de la secuencia para que el modelo ignore el padding.


In [28]:
genre_seq_feat = VarLenSparseFeat(
    SparseFeat("genres_seq", vocabulary_size=VOCAB_SIZE_GENRE, embedding_dim=16),
    maxlen=MAXLEN,
    combiner="mean",
    length_name="genres_seq_len"
)

linear_feature_columns = [user_feat, item_feat, genre_seq_feat]
dnn_feature_columns    = [user_feat, item_feat, genre_seq_feat]

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

Se define la función **`build_X`**, que prepara los datos de entrada en el formato requerido por *DeepCTR-torch*.

El resultado es un diccionario `X` que contiene todos los datos en el formato esperado por el modelo DeepFM, listo para ser usado en el entrenamiento y la evaluación.


In [29]:
def build_X(df_):
    X = {}

    for name in feature_names:
        if name in df_.columns:
            X[name] = df_[name].values

    X["genres_seq"]     = np.stack(df_["genres_seq"].values)
    X["genres_seq_len"] = df_["genres_seq_len"].values

    return X

Se sigue el mismo proceso que al entrenar sin género

In [30]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
val_df, test_df  = train_test_split(val_df, test_size=0.5, random_state=42, stratify=val_df["label"])

In [31]:
X_train, y_train = build_X(train_df), train_df["label"].values
X_val, y_val = build_X(val_df), val_df["label"].values
X_test, y_test = build_X(test_df), test_df["label"].values

In [32]:
model = DeepFM(
    linear_feature_columns, dnn_feature_columns,
    task='binary', dnn_hidden_units=(128,64), dnn_dropout=0.2, device="cpu"
)

In [33]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

In [34]:
model.fit(X_train, y_train, batch_size=2048, epochs=10, verbose=2, validation_data=(X_val, y_val))

cpu
Train on 80000 samples, validate on 10000 samples, 40 steps per epoch
Epoch 1/10
2s - loss:  0.6817 - auc:  0.6435 - val_auc:  0.7481
Epoch 2/10
1s - loss:  0.6280 - auc:  0.7683 - val_auc:  0.7617
Epoch 3/10
1s - loss:  0.5772 - auc:  0.7810 - val_auc:  0.7663
Epoch 4/10
1s - loss:  0.5600 - auc:  0.7888 - val_auc:  0.7684
Epoch 5/10
2s - loss:  0.5519 - auc:  0.7943 - val_auc:  0.7700
Epoch 6/10
1s - loss:  0.5478 - auc:  0.7955 - val_auc:  0.7705
Epoch 7/10
2s - loss:  0.5454 - auc:  0.7971 - val_auc:  0.7710
Epoch 8/10
1s - loss:  0.5438 - auc:  0.7962 - val_auc:  0.7716
Epoch 9/10
1s - loss:  0.5426 - auc:  0.7972 - val_auc:  0.7710
Epoch 10/10
1s - loss:  0.5418 - auc:  0.7971 - val_auc:  0.7719


In [35]:
y_scores = model.predict(X_test, batch_size=4096).reshape(-1)
test_eval = test_df[["user_id","item_id","label"]].copy()
test_eval["score"] = y_scores

In [36]:
K_list = [5, 10, 20]
rows = []
for K in K_list:
    p, r = macro_precision_recall_at_k(test_eval, K)
    rows.append({
        'K': K,
        'Precision@K (macro)': p,
        'Recall@K (macro)': r,
    })

metrics_df = pd.DataFrame(rows)
metrics_df

,K,Precision@K (macro),Recall@K (macro)
0,5,0.559529,0.704518
1,10,0.417559,0.870710
2,20,0.270557,0.970563


# 5. Actividad

# Parte 1 (2 puntos)

Explique por qué la incorporación de los géneros produce una mejoría casi nula en las métricas.

# Parte 2 (2 puntos)

Lea el abstract y la introducción del paper de [DeepFM](https://arxiv.org/abs/1703.04247). Con esta información, explique con sus palabras de qué se trata el modelo DeepFM y cuál es su relación con las FM tradicionales

# Parte 3 (4 puntos)

Elija una de las dos versiones de DeepFM entrenadas (con o sin géneros de películas). Modifique los parámetros "batch size" y/o "epochs". Intente al menos 3 combinaciones y explique la diferencia entre el desempeño (en caso de no haber diferencia, explique la causa)

# Parte 4 (2 puntos)

Con la mejor combinación de parámetros obtenida en el ítem anterior, modifique la compilación del modelo para incluir todas las métricas disponibles*.  

Para ello, cambie el parámetro `metrics` en el método `model.compile`